<a href="https://colab.research.google.com/github/davis-mironga/marsabit-ecosystem-analysis/blob/main/01_GEE_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📓 Notebook 1 — GEE Data Preprocessing
## The Marsabit Footprint: A 36-Year Data Study on Livestock Pressure and Ecosystem Health

---

**Purpose of this Notebook:**
This is the foundation notebook. Everything produced here feeds into Notebooks 2, 3, and 4.
We use Google Earth Engine (GEE) because satellite imagery is massive — it lives in Google's cloud and we process it there before exporting only what we need.

**What this notebook produces:**
- ✅ NDVI layers for 5 time periods (1990, 2000, 2010, 2020, 2024)
- ✅ CHIRPS Rainfall data (climate control variable)
- ✅ SRTM Elevation data (terrain control variable)
- ✅ Distance-to-Water raster (livestock proxy for LCI)
- ✅ All exported to Google Drive as GeoTIFFs

## ⚙️ STEP 1 — Install Libraries & Authenticate with Google Earth Engine

**What we are doing:**
Before anything else, we need two things:
1. `geemap` — a Python library that lets us control GEE from this notebook and visualize maps interactively.
2. GEE Authentication — GEE needs to verify you have permission to use its satellite data. This opens a Google login link. After logging in, paste the verification code back here.

> ⚠️ When `ee.Authenticate()` runs, click the link, sign in with your GEE-registered Google account, and paste the authorization code into the box.

In [1]:
# Install geemap (only needed once per Colab session)
!pip install geemap -q

import ee
import geemap

# Authenticate your Google account with GEE
# A link will appear — log in and paste the code back here
ee.Authenticate()

# Initialize GEE with your Google Cloud Project
ee.Initialize(project='mironga-project-marsabit')

print("✅ GEE authenticated and initialized successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.0 MB/s eta 0:00:00
✅ GEE authenticated and initialized successfully!


## 🗺️ STEP 2 — Define the Study Area (Marsabit County Boundary)

**What we are doing:**
We load the official administrative boundary of Marsabit County from FAO's Global Administrative
Unit Layers (GAUL) dataset hosted directly inside GEE. This boundary acts as our "cookie cutter"
— every satellite image we load will be clipped to this shape.

**Why FAO GAUL?**
It is an internationally recognised boundary dataset available directly inside GEE — no file
uploads needed.

**What the map will show:**
An interactive map with the Marsabit County boundary outlined in red, so we can confirm we have
the correct area before doing any analysis.

**Output of this step:**
- `marsabit_roi` → The full FeatureCollection (used for filtering imagery)
- `marsabit_geom` → Just the geometry/shape (used for clipping images)

In [2]:
# Load Marsabit County boundary from FAO GAUL Level 2 (district/county level)
marsabit_roi = (
    ee.FeatureCollection("FAO/GAUL/2015/level2")
    .filter(ee.Filter.eq('ADM2_NAME', 'Marsabit'))
)

# Extract just the geometry — used for clipping images later
marsabit_geom = marsabit_roi.geometry()

# Confirm it loaded correctly
area_km2 = marsabit_geom.area().divide(1e6).getInfo()
print(f"✅ Marsabit County boundary loaded successfully.")
print(f"   📐 Area: {area_km2:,.0f} km²")

# Visualize on an interactive map
Map = geemap.Map()
Map.centerObject(marsabit_roi, 8)
Map.addLayer(
    marsabit_roi,
    {'color': 'red'},
    'Marsabit County Boundary'
)
Map.addLayerControl()
Map

✅ Marsabit County boundary loaded successfully.
   📐 Area: 61,251 km²


Map(center=[2.9274372660911148, 37.48368008533612], controls=(WidgetControl(options=['position', 'transparent_…

## 📡 STEP 3 — Define the Satellite Image Loading Function

**What we are doing:**
We write one reusable function that loads Landsat imagery correctly for any time period.
This function handles three critical processing steps required by our project methodology:

1. **Scaling Factors** — Landsat Collection 2 Level 2 stores values as integers for storage
   efficiency. We must multiply by `0.0000275` and add `-0.2` to convert them into real surface
   reflectance values (0 to 1 range). Without this, NDVI values are scientifically meaningless.

2. **Cloud Masking** — We use each image's built-in QA_PIXEL band to automatically remove
   cloudy and cloud-shadow pixels before computing the median composite.

3. **Multi-Year Compositing** — Instead of one single year, we use 5-year windows
   (e.g., 1988–1992 for the "1990" period). Taking the median across many images removes
   remaining artifacts and reduces the effect of unusual rainfall years — exactly as specified
   in our project documentation.

**The 5 time periods we will process:**

| Label | Date Window | Satellite | Sensor Code |
|-------|-------------|-----------|-------------|
| 1990  | 1988–1992   | Landsat 5 | L57         |
| 2000  | 1998–2002   | Landsat 5 | L57         |
| 2010  | 2008–2012   | Landsat 7 | L57         |
| 2020  | 2018–2022   | Landsat 8 | L89         |
| 2024  | 2022–2024   | Landsat 9 | L89         |

> ℹ️ **Band Note:** Landsat 5/7 use B4 (NIR) and B3 (Red) for NDVI.
> Landsat 8/9 use B5 (NIR) and B4 (Red). Our function handles this automatically
> using the `sensor` parameter.

In [3]:
# ─────────────────────────────────────────────────────────────────────
# FUNCTION 1: Apply Landsat Collection 2 Level 2 Scale Factors
# Converts raw integer DN values → real surface reflectance (0 to 1)
# ─────────────────────────────────────────────────────────────────────
def apply_scale_factors(image):
    optical_bands = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    return image.addBands(optical_bands, None, True)


# ─────────────────────────────────────────────────────────────────────
# FUNCTION 2: Cloud Mask using the QA_PIXEL band
# Removes cloud pixels (bit 5) and cloud shadow pixels (bit 3)
# ─────────────────────────────────────────────────────────────────────
def mask_landsat_clouds(image):
    qa = image.select('QA_PIXEL')
    cloud_shadow = qa.bitwiseAnd(1 << 3).eq(0)  # Bit 3 = Cloud Shadow
    cloud        = qa.bitwiseAnd(1 << 5).eq(0)  # Bit 5 = Cloud
    return image.updateMask(cloud_shadow.And(cloud))


# ─────────────────────────────────────────────────────────────────────
# FUNCTION 3: Build NDVI Composite for a given time period
# This is the main function — we call it 5 times, once per decade
# ─────────────────────────────────────────────────────────────────────
def get_ndvi_composite(collection_id, start_year, end_year, roi, sensor='L57'):
    """
    Loads a multi-year Landsat median composite with:
    - Cloud masking applied
    - Scale factors applied
    - NDVI calculated and returned

    Args:
        collection_id : GEE Landsat collection string
        start_year    : Start year as string e.g. '1988'
        end_year      : End year as string e.g. '1992'
        roi           : GEE geometry to clip imagery to
        sensor        : 'L57' for Landsat 5 & 7 | 'L89' for Landsat 8 & 9

    Returns:
        Single-band NDVI image clipped to roi
    """
    collection = (
        ee.ImageCollection(collection_id)
        .filterBounds(roi)
        .filterDate(f'{start_year}-01-01', f'{end_year}-12-31')
        .map(mask_landsat_clouds)
        .map(apply_scale_factors)
    )

    composite = collection.median().clip(roi)

    # NDVI bands differ between sensor generations
    if sensor == 'L57':
        nir, red = 'SR_B4', 'SR_B3'   # Landsat 5 and 7
    else:
        nir, red = 'SR_B5', 'SR_B4'   # Landsat 8 and 9

    ndvi = composite.normalizedDifference([nir, red]).rename('NDVI')
    return ndvi


print("✅ Three functions defined and ready:")
print("   → apply_scale_factors()  : Raw DN → surface reflectance")
print("   → mask_landsat_clouds()  : Removes cloud & shadow pixels")
print("   → get_ndvi_composite()   : Builds multi-year NDVI composite")

✅ Three functions defined and ready:
   → apply_scale_factors()  : Raw DN → surface reflectance
   → mask_landsat_clouds()  : Removes cloud & shadow pixels
   → get_ndvi_composite()   : Builds multi-year NDVI composite


## 🌿 STEP 4 — Compute NDVI for All 5 Time Periods

**What we are doing:**
We now call the function from Step 3 five times — once per decade. Each call automatically
processes hundreds of satellite images, applies cloud masking, applies scaling, and returns
one clean NDVI image representing that entire era.

**NDVI Value Interpretation:**
| NDVI Range | What it means on the ground |
|------------|------------------------------|
| 0.6 – 1.0  | Dense healthy vegetation (Mt. Marsabit Forest) |
| 0.3 – 0.6  | Moderate vegetation / rangeland |
| 0.1 – 0.3  | Sparse vegetation / degraded rangeland |
| 0.0 – 0.1  | Bare soil, rock, heavily degraded land |
| Below 0    | Water bodies |

**What the map will show:**
All 5 NDVI layers loaded into one interactive map. Use the layer control panel
(top right) to toggle each year on and off to visually compare vegetation across decades.

> 💡 **Tip:** Start with 1990 ON and 2024 ON, all others OFF — this gives you the
> clearest before-and-after picture of 36 years of change.

In [4]:
# ─────────────────────────────────────────────────────────────────────
# Compute NDVI composites for all 5 time periods
# GEE processes these lazily — computation happens when the map renders
# ─────────────────────────────────────────────────────────────────────

print("⏳ Building NDVI composites for all 5 periods...")

ndvi_1990 = get_ndvi_composite(
    'LANDSAT/LT05/C02/T1_L2', '1988', '1992', marsabit_geom, sensor='L57'
)
print("   ✓ 1990 period (1988–1992, Landsat 5)")

ndvi_2000 = get_ndvi_composite(
    'LANDSAT/LT05/C02/T1_L2', '1998', '2002', marsabit_geom, sensor='L57'
)
print("   ✓ 2000 period (1998–2002, Landsat 5)")

ndvi_2010 = get_ndvi_composite(
    'LANDSAT/LE07/C02/T1_L2', '2008', '2012', marsabit_geom, sensor='L57'
)
print("   ✓ 2010 period (2008–2012, Landsat 7)")

ndvi_2020 = get_ndvi_composite(
    'LANDSAT/LC08/C02/T1_L2', '2018', '2022', marsabit_geom, sensor='L89'
)
print("   ✓ 2020 period (2018–2022, Landsat 8)")

ndvi_2024 = get_ndvi_composite(
    'LANDSAT/LC09/C02/T1_L2', '2022', '2024', marsabit_geom, sensor='L89'
)
print("   ✓ 2024 period (2022–2024, Landsat 9)")

# ─────────────────────────────────────────────────────────────────────
# Visualization parameters
# Red = degraded/bare land | Green = healthy vegetation
# ─────────────────────────────────────────────────────────────────────
ndvi_vis = {
    'min': -0.2,
    'max': 0.8,
    'palette': ['#d73027','#f46d43','#fdae61','#fee08b',
                '#d9ef8b','#a6d96a','#66bd63','#1a9850']
}

# Build the interactive map
Map2 = geemap.Map()
Map2.centerObject(marsabit_roi, 8)
Map2.addLayer(ndvi_1990, ndvi_vis, 'NDVI 1990')
Map2.addLayer(ndvi_2000, ndvi_vis, 'NDVI 2000', False)  # False = off by default
Map2.addLayer(ndvi_2010, ndvi_vis, 'NDVI 2010', False)
Map2.addLayer(ndvi_2020, ndvi_vis, 'NDVI 2020', False)
Map2.addLayer(ndvi_2024, ndvi_vis, 'NDVI 2024')
Map2.addLayer(marsabit_roi, {'color': 'white'}, 'Marsabit Boundary')
Map2.addLayerControl()

print("\n✅ All 5 NDVI composites ready!")
print("   Toggle layers on the map to compare decades.")
print("   Red = bare/degraded | Green = healthy vegetation")
Map2

⏳ Building NDVI composites for all 5 periods...
   ✓ 1990 period (1988–1992, Landsat 5)
   ✓ 2000 period (1998–2002, Landsat 5)
   ✓ 2010 period (2008–2012, Landsat 7)
   ✓ 2020 period (2018–2022, Landsat 8)
   ✓ 2024 period (2022–2024, Landsat 9)

✅ All 5 NDVI composites ready!
   Toggle layers on the map to compare decades.
   Red = bare/degraded | Green = healthy vegetation


Map(center=[2.927437266091119, 37.48368008533611], controls=(WidgetControl(options=['position', 'transparent_b…

## 📉 STEP 5 — Compute NDVI Change Maps (Decadal Differences)

**What we are doing:**
We subtract NDVI values between time periods to create NDVI change maps.
These show us exactly where vegetation has increased (greening) or
decreased (degradation) over each decade.

**How to read the change map:**
| Colour | NDVI Change Value | Meaning |
|--------|------------------|---------|
| 🔴 Red | Negative values | Vegetation LOSS — land became more bare |
| 🟡 Yellow | Near zero | Little to no change |
| 🟢 Green | Positive values | Vegetation GAIN — land recovered |

**The 4 decadal change maps we compute:**
| Change Map | Period | What it captures |
|------------|--------|-----------------|
| Change 1 | 1990 → 2000 | First decade of change |
| Change 2 | 2000 → 2010 | Second decade of change |
| Change 3 | 2010 → 2020 | Third decade of change |
| Change 4 | 2020 → 2024 | Most recent change |

**The most important output — Total Change (1990 → 2024):**
This is the full 36-year change map. It becomes the dependent variable Y
in our regression model in Notebook 4:

> NDVI_change = β₀ + β₁(LCI) + β₂(Rainfall) + β₃(Elevation) + ε

In [5]:
# ─────────────────────────────────────────────────────────────────────
# Compute NDVI change between each consecutive time period
# Formula: Change = Later period NDVI - Earlier period NDVI
# Positive = greening | Negative = degradation
# ─────────────────────────────────────────────────────────────────────

ndvi_change_90_00 = (ndvi_2000.subtract(ndvi_1990)
                    .rename('NDVI_change_1990_2000'))

ndvi_change_00_10 = (ndvi_2010.subtract(ndvi_2000)
                    .rename('NDVI_change_2000_2010'))

ndvi_change_10_20 = (ndvi_2020.subtract(ndvi_2010)
                    .rename('NDVI_change_2010_2020'))

ndvi_change_20_24 = (ndvi_2024.subtract(ndvi_2020)
                    .rename('NDVI_change_2020_2024'))

# Total change: 1990 → 2024 — this is our Y variable for regression
ndvi_change_total = (ndvi_2024.subtract(ndvi_1990)
                    .rename('NDVI_change_total_1990_2024'))

# ─────────────────────────────────────────────────────────────────────
# Visualization parameters for change maps
# Red = loss | Yellow = no change | Green = gain
# ─────────────────────────────────────────────────────────────────────
change_vis = {
    'min': -0.4,
    'max': 0.4,
    'palette': ['#d73027','#f46d43','#fdae61',
                '#ffffbf',
                '#a6d96a','#66bd63','#1a9850']
}

# ─────────────────────────────────────────────────────────────────────
# Visualize all change maps on an interactive map
# ─────────────────────────────────────────────────────────────────────
Map3 = geemap.Map()
Map3.centerObject(marsabit_roi, 8)

Map3.addLayer(ndvi_change_90_00, change_vis, 'Change 1990→2000')
Map3.addLayer(ndvi_change_00_10, change_vis, 'Change 2000→2010', False)
Map3.addLayer(ndvi_change_10_20, change_vis, 'Change 2010→2020', False)
Map3.addLayer(ndvi_change_20_24, change_vis, 'Change 2020→2024', False)
Map3.addLayer(ndvi_change_total, change_vis, 'TOTAL Change 1990→2024')
Map3.addLayer(marsabit_roi, {'color': 'white'}, 'Marsabit Boundary')
Map3.addLayerControl()

print("✅ NDVI Change Maps computed:")
print("   → ndvi_change_90_00  : Change 1990 to 2000")
print("   → ndvi_change_00_10  : Change 2000 to 2010")
print("   → ndvi_change_10_20  : Change 2010 to 2020")
print("   → ndvi_change_20_24  : Change 2020 to 2024")
print("   → ndvi_change_total  : TOTAL change 1990 to 2024 (Y variable)")
print("\n💡 Tip: Turn on TOTAL Change 1990→2024 layer for the full picture")
Map3

✅ NDVI Change Maps computed:
   → ndvi_change_90_00  : Change 1990 to 2000
   → ndvi_change_00_10  : Change 2000 to 2010
   → ndvi_change_10_20  : Change 2010 to 2020
   → ndvi_change_20_24  : Change 2020 to 2024
   → ndvi_change_total  : TOTAL change 1990 to 2024 (Y variable)

💡 Tip: Turn on TOTAL Change 1990→2024 layer for the full picture


Map(center=[2.9274372660911148, 37.48368008533612], controls=(WidgetControl(options=['position', 'transparent_…

## 🌧️ STEP 6 — Load CHIRPS Rainfall Data (Climate Control Variable)

**What we are doing:**
We load rainfall data from CHIRPS (Climate Hazards Group InfraRed Precipitation with
Station data). Rainfall is a critical control variable in our regression model because
vegetation growth is heavily influenced by how much rain an area receives.

**Why we need rainfall as a control variable:**
Without controlling for rainfall, we cannot isolate the effect of livestock pressure
on vegetation. An area might have low NDVI simply because it receives very little rain
— not because of livestock. By including rainfall in our regression, we separate:
- Vegetation loss due to → CLIMATE (rainfall variability)
- Vegetation loss due to → LIVESTOCK PRESSURE (what we want to measure)

**What CHIRPS is:**
- Global rainfall dataset at 5km resolution
- Available from 1981 to present
- Combines satellite imagery with ground station data
- Widely used in dryland ecosystem research in Africa

**What we compute:**
We calculate the MEAN ANNUAL RAINFALL for each of our 5 time periods,
matching exactly the same date windows used for NDVI.

**Processing steps:**
1. Load CHIRPS daily data for each period
2. Sum daily rainfall to get annual totals
3. Average across the multi-year window
4. Resample from 5km → 30m to match Landsat resolution
5. Apply Z-score normalization for use in regression

In [6]:
# ─────────────────────────────────────────────────────────────────────
# FUNCTION: Compute mean annual rainfall for a given period
# CHIRPS data is daily — we sum to annual then average across years
# ─────────────────────────────────────────────────────────────────────
def get_chirps_rainfall(start_year, end_year, roi):
    """
    Computes mean annual rainfall (mm) for a multi-year period.

    Args:
        start_year : Start year as string e.g. '1988'
        end_year   : End year as string e.g. '1992'
        roi        : GEE geometry to clip to

    Returns:
        Single-band rainfall image clipped to roi (mean annual mm)
    """
    chirps = (
        ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
        .filterBounds(roi)
        .filterDate(f'{start_year}-01-01', f'{end_year}-12-31')
        .select('precipitation')
    )

    # Sum daily rainfall to get total for the whole period
    total_rainfall = chirps.sum()

    # Divide by number of years to get mean annual rainfall
    n_years = int(end_year) - int(start_year) + 1
    mean_annual = total_rainfall.divide(n_years)

    # Resample from 5km to 30m to match Landsat resolution
    mean_annual_30m = (mean_annual
                       .resample('bilinear')
                       .reproject(crs='EPSG:4326', scale=30))

    return mean_annual_30m.clip(roi).rename('rainfall_mm')


# ─────────────────────────────────────────────────────────────────────
# Compute rainfall for all 5 time periods
# Using same date windows as NDVI composites
# ─────────────────────────────────────────────────────────────────────
print("⏳ Computing CHIRPS rainfall for all 5 periods...")

rainfall_1990 = get_chirps_rainfall('1988', '1992', marsabit_geom)
print("   ✓ Rainfall 1990 period (1988–1992)")

rainfall_2000 = get_chirps_rainfall('1998', '2002', marsabit_geom)
print("   ✓ Rainfall 2000 period (1998–2002)")

rainfall_2010 = get_chirps_rainfall('2008', '2012', marsabit_geom)
print("   ✓ Rainfall 2010 period (2008–2012)")

rainfall_2020 = get_chirps_rainfall('2018', '2022', marsabit_geom)
print("   ✓ Rainfall 2020 period (2018–2022)")

rainfall_2024 = get_chirps_rainfall('2022', '2024', marsabit_geom)
print("   ✓ Rainfall 2024 period (2022–2024)")

# ─────────────────────────────────────────────────────────────────────
# Compute rainfall CHANGE between 1990 and 2024
# Used as control variable in regression
# ─────────────────────────────────────────────────────────────────────
rainfall_change = (rainfall_2024.subtract(rainfall_1990)
                  .rename('rainfall_change_1990_2024'))

# ─────────────────────────────────────────────────────────────────────
# Visualize rainfall on an interactive map
# Blue = high rainfall | White/Yellow = low rainfall
# ─────────────────────────────────────────────────────────────────────
rainfall_vis = {
    'min': 200,
    'max': 800,
    'palette': ['#ffffcc','#a1dab4','#41b6c4','#2c7fb8','#253494']
}

Map4 = geemap.Map()
Map4.centerObject(marsabit_roi, 8)
Map4.addLayer(rainfall_1990, rainfall_vis, 'Rainfall 1990')
Map4.addLayer(rainfall_2024, rainfall_vis, 'Rainfall 2024', False)
Map4.addLayer(marsabit_roi, {'color': 'red'}, 'Marsabit Boundary')
Map4.addLayerControl()

print("\n✅ CHIRPS Rainfall layers ready!")
print("   → rainfall_1990 to rainfall_2024  : Mean annual rainfall per period")
print("   → rainfall_change                 : Rainfall change 1990→2024")
print("\n💡 Tip: Blue = high rainfall (Mt. Marsabit area)")
print("        Yellow = low rainfall (desert margins near Turkana)")
Map4

⏳ Computing CHIRPS rainfall for all 5 periods...
   ✓ Rainfall 1990 period (1988–1992)
   ✓ Rainfall 2000 period (1998–2002)
   ✓ Rainfall 2010 period (2008–2012)
   ✓ Rainfall 2020 period (2018–2022)
   ✓ Rainfall 2024 period (2022–2024)

✅ CHIRPS Rainfall layers ready!
   → rainfall_1990 to rainfall_2024  : Mean annual rainfall per period
   → rainfall_change                 : Rainfall change 1990→2024

💡 Tip: Blue = high rainfall (Mt. Marsabit area)
        Yellow = low rainfall (desert margins near Turkana)


Map(center=[2.9274372660911148, 37.48368008533612], controls=(WidgetControl(options=['position', 'transparent_…

## 🏔️ STEP 7 — Load SRTM Elevation Data (Terrain Control Variable)

**What we are doing:**
We load elevation data from the SRTM (Shuttle Radar Topography Mission) dataset.
Elevation is our second control variable in the regression model.

**Why elevation matters:**
Elevation controls vegetation in two important ways:
1. Higher elevation = cooler temperatures + more rainfall = more vegetation
   (Mt. Marsabit at ~1700m has montane forest because of this)
2. Lower elevation = hotter + drier = sparse vegetation or desert
   (The lowland desert margins near Lake Turkana sit at ~400m)

Without controlling for elevation, we might wrongly blame livestock for
vegetation differences that are actually caused by terrain.

**What SRTM is:**
- Captured by NASA Space Shuttle in February 2000
- Global coverage at 30m resolution
- The standard elevation dataset used in GIS and remote sensing worldwide

**What we compute:**
From the raw elevation (DEM) we also derive:
- Slope — steeper slopes affect where livestock can graze
- These become additional spatial context layers

**Our regression equation with all control variables:**
> NDVI_change = β₀ + β₁(LCI) + β₂(Rainfall) + β₃(Elevation) + ε

In [7]:
# ─────────────────────────────────────────────────────────────────────
# Load SRTM Digital Elevation Model (DEM)
# Resolution: 30m — matches our Landsat imagery perfectly
# ─────────────────────────────────────────────────────────────────────
srtm = ee.Image("USGS/SRTMGL1_003")

# Clip to Marsabit boundary
elevation = srtm.select('elevation').clip(marsabit_geom)

# ─────────────────────────────────────────────────────────────────────
# Derive Slope from elevation
# Slope tells us terrain steepness — affects where livestock can graze
# ─────────────────────────────────────────────────────────────────────
slope = ee.Terrain.slope(srtm).clip(marsabit_geom)

# ─────────────────────────────────────────────────────────────────────
# Get basic elevation statistics for Marsabit
# ─────────────────────────────────────────────────────────────────────
stats = elevation.reduceRegion(
    reducer=ee.Reducer.minMax().combine(
        ee.Reducer.mean(), sharedInputs=True
    ),
    geometry=marsabit_geom,
    scale=30,
    maxPixels=1e9
)

stats_dict = stats.getInfo()
print("✅ SRTM Elevation loaded successfully!")
print(f"   📐 Elevation statistics for Marsabit County:")
print(f"   → Minimum elevation : {stats_dict['elevation_min']:.0f} m")
print(f"   → Maximum elevation : {stats_dict['elevation_max']:.0f} m")
print(f"   → Mean elevation    : {stats_dict['elevation_mean']:.0f} m")

# ─────────────────────────────────────────────────────────────────────
# Visualization parameters
# ─────────────────────────────────────────────────────────────────────
elevation_vis = {
    'min': 300,
    'max': 1800,
    'palette': ['#313695','#4575b4','#74add1','#abd9e9',
                '#e0f3f8','#ffffbf','#fee090','#fdae61',
                '#f46d43','#d73027','#a50026']
}

slope_vis = {
    'min': 0,
    'max': 30,
    'palette': ['white', 'yellow', 'orange', 'red']
}

# ─────────────────────────────────────────────────────────────────────
# Visualize on interactive map
# ─────────────────────────────────────────────────────────────────────
Map5 = geemap.Map()
Map5.setCenter(37.9, 2.3, 8)
Map5.addLayer(elevation, elevation_vis, 'Elevation (m)')
Map5.addLayer(slope, slope_vis, 'Slope (degrees)', False)
Map5.addLayer(marsabit_roi, {'color': 'white'}, 'Marsabit Boundary')
Map5.addLayerControl()

print("\n💡 Tip: The bright peak in the centre is Mt. Marsabit (~1700m)")
print("        Dark blue = lowlands | Red/white = highlands")
Map5

✅ SRTM Elevation loaded successfully!
   📐 Elevation statistics for Marsabit County:
   → Minimum elevation : 318 m
   → Maximum elevation : 2276 m
   → Mean elevation    : 596 m

💡 Tip: The bright peak in the centre is Mt. Marsabit (~1700m)
        Dark blue = lowlands | Red/white = highlands


Map(center=[2.3, 37.9], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(c…

## 💧 STEP 8 — Build the Livestock Concentration Index (LCI)

**What we are doing:**
We build the Livestock Concentration Index (LCI) — the most important variable
in this entire project. LCI is our proxy for livestock pressure since we do not
have direct livestock census data at fine spatial resolution.

**The core idea:**
In arid rangelands like Marsabit, livestock movement is dictated by water.
Animals concentrate around water points (boreholes, rivers, pans) and spread
outward as water becomes scarcer. This creates a predictable spatial pattern:

> The closer to a water point → the higher the livestock pressure
> The further from a water point → the lower the livestock pressure

**The LCI Formula from our project documentation:**
LCI = α(1/Distance_to_Water) + β(Settlement_Density)

**In this step we compute the Distance-to-Water component:**
1. Load known water points for Marsabit (boreholes, pans, rivers)
2. Compute Euclidean distance from every pixel to the nearest water point
3. Invert the distance (1/distance) so pixels NEAR water get HIGH values
4. Normalize to 0-1 scale

**Why this matters scientifically:**
If our analysis shows that high LCI areas have lower NDVI change (more
degradation), that is statistical evidence that livestock pressure — not
just climate — is driving vegetation loss in Marsabit.

**Data sources for water points:**
- JRC Global Surface Water (permanent water bodies)
- OpenStreetMap water points via GEE
- These act as proxies until actual WRA/County borehole data is obtained

In [8]:
# ─────────────────────────────────────────────────────────────────────
# STEP 8A: Load Water Bodies from JRC Global Surface Water dataset
# This gives us permanent water features — rivers, pans, lakes
# These are the places livestock concentrate around
# ─────────────────────────────────────────────────────────────────────

# Load JRC Global Surface Water — occurrence band (0-100%)
# We keep pixels with water occurrence > 50% (permanent/semi-permanent water)
jrc_water = (
    ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
    .select('occurrence')
    .gte(50)  # Keep pixels that are water at least 50% of the time
    .clip(marsabit_geom)
)

# Convert water pixels to vectors (points) for distance calculation
water_vectors = jrc_water.selfMask().reduceToVectors(
    geometry=marsabit_geom,
    scale=500,           # 500m scale for efficiency
    maxPixels=1e9,
    geometryType='centroid'
)

print(f"✅ Water features loaded")
print(f"   → Water bodies found: {water_vectors.size().getInfo()} features")

# ─────────────────────────────────────────────────────────────────────
# STEP 8B: Compute Distance to Nearest Water Point
# For every pixel in Marsabit, calculate distance to nearest water body
# ─────────────────────────────────────────────────────────────────────

# Distance transform — every pixel gets distance to nearest water (metres)
distance_to_water = (
    jrc_water.fastDistanceTransform(neighborhood=2048)
    .sqrt()
    .multiply(ee.Image.pixelArea().sqrt())  # Convert to metres
    .clip(marsabit_geom)
    .rename('distance_to_water_m')
)

# ─────────────────────────────────────────────────────────────────────
# STEP 8C: Build the LCI Distance Component
# Invert distance: pixels NEAR water = HIGH livestock pressure
# Apply min-max normalization to scale between 0 and 1
# ─────────────────────────────────────────────────────────────────────

# Invert: 1/distance (add small value to avoid division by zero)
inverse_distance = (
    ee.Image(1)
    .divide(distance_to_water.add(1))
    .rename('inverse_distance')
)

# Min-max normalization to 0-1 scale
dist_min_max = inverse_distance.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=marsabit_geom,
    scale=500,
    maxPixels=1e9
)

dist_min = ee.Number(dist_min_max.get('inverse_distance_min'))
dist_max = ee.Number(dist_min_max.get('inverse_distance_max'))

# Normalized LCI distance component (0 = far from water, 1 = at water)
lci_distance = (
    inverse_distance.subtract(dist_min)
    .divide(dist_max.subtract(dist_min))
    .rename('LCI_distance_component')
)

# ─────────────────────────────────────────────────────────────────────
# STEP 8D: Visualize Distance to Water and LCI component
# ─────────────────────────────────────────────────────────────────────
distance_vis = {
    'min': 0,
    'max': 50000,  # 0 to 50km
    'palette': ['#08306b','#2171b5','#6baed6','#bdd7e7','#eff3ff','white']
}

lci_vis = {
    'min': 0,
    'max': 1,
    'palette': ['white', '#ffffb2', '#fecc5c', '#fd8d3c', '#e31a1c']
}

Map6 = geemap.Map()
Map6.setCenter(37.9, 2.3, 8)
Map6.addLayer(distance_to_water, distance_vis, 'Distance to Water (m)')
Map6.addLayer(lci_distance, lci_vis, 'LCI Distance Component', False)
Map6.addLayer(jrc_water.selfMask(),
              {'palette': ['blue']}, 'Water Bodies')
Map6.addLayer(marsabit_roi, {'color': 'white'}, 'Marsabit Boundary')
Map6.addLayerControl()

print("✅ Livestock Concentration Index — Distance Component ready!")
print("   → distance_to_water  : Distance in metres to nearest water body")
print("   → lci_distance       : Normalized LCI (0=far from water, 1=at water)")
print("\n💡 On the LCI map:")
print("   Red/Orange = HIGH livestock pressure (near water)")
print("   White      = LOW livestock pressure (far from water)")
Map6

✅ Water features loaded
   → Water bodies found: 8 features
✅ Livestock Concentration Index — Distance Component ready!
   → distance_to_water  : Distance in metres to nearest water body
   → lci_distance       : Normalized LCI (0=far from water, 1=at water)

💡 On the LCI map:
   Red/Orange = HIGH livestock pressure (near water)
   White      = LOW livestock pressure (far from water)


Map(center=[2.3, 37.9], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(c…

In [9]:
# ─────────────────────────────────────────────────────────────────────
# Export configuration — GEE Assets
# ─────────────────────────────────────────────────────────────────────
ASSET_PATH = 'projects/mironga-project-marsabit/assets/marsabit'
SCALE = 30
CRS = 'EPSG:4326'

# ─────────────────────────────────────────────────────────────────────
# STEP 9A: Create the asset folder first
# Run this ONCE — comment it out after first run
# ─────────────────────────────────────────────────────────────────────
try:
    ee.data.createAsset(
        {'type': 'Folder'},
        ASSET_PATH
    )
    print(f"✅ Asset folder created: {ASSET_PATH}")
except Exception as e:
    print(f"ℹ️  Folder may already exist: {e}")

# ─────────────────────────────────────────────────────────────────────
# HELPER FUNCTION: Export image to GEE Asset
# ─────────────────────────────────────────────────────────────────────
def export_to_asset(image, asset_name, description):
    """
    Submits a GEE export task to GEE Assets.

    Args:
        image      : GEE image to export
        asset_name : Asset filename (no path needed)
        description: Task description in GEE Task Manager
    """
    asset_id = f'{ASSET_PATH}/{asset_name}'

    task = ee.batch.Export.image.toAsset(
        image=image,
        description=description,
        assetId=asset_id,
        region=marsabit_geom,
        scale=SCALE,
        crs=CRS,
        maxPixels=1e13
    )
    task.start()
    return task


# ─────────────────────────────────────────────────────────────────────
# Submit all export tasks to GEE Assets
# ─────────────────────────────────────────────────────────────────────
print("⏳ Submitting export tasks to GEE Assets...")
print(f"   Path: {ASSET_PATH}/\n")

# --- NDVI Composites ---
export_to_asset(ndvi_1990, 'ndvi_1990', 'NDVI_1990_Marsabit')
print("   ✓ Task submitted: ndvi_1990")

export_to_asset(ndvi_2000, 'ndvi_2000', 'NDVI_2000_Marsabit')
print("   ✓ Task submitted: ndvi_2000")

export_to_asset(ndvi_2010, 'ndvi_2010', 'NDVI_2010_Marsabit')
print("   ✓ Task submitted: ndvi_2010")

export_to_asset(ndvi_2020, 'ndvi_2020', 'NDVI_2020_Marsabit')
print("   ✓ Task submitted: ndvi_2020")

export_to_asset(ndvi_2024, 'ndvi_2024', 'NDVI_2024_Marsabit')
print("   ✓ Task submitted: ndvi_2024")

# --- NDVI Change Maps ---
export_to_asset(ndvi_change_90_00,
                'ndvi_change_1990_2000', 'NDVI_Change_1990_2000')
print("   ✓ Task submitted: ndvi_change_1990_2000")

export_to_asset(ndvi_change_00_10,
                'ndvi_change_2000_2010', 'NDVI_Change_2000_2010')
print("   ✓ Task submitted: ndvi_change_2000_2010")

export_to_asset(ndvi_change_10_20,
                'ndvi_change_2010_2020', 'NDVI_Change_2010_2020')
print("   ✓ Task submitted: ndvi_change_2010_2020")

export_to_asset(ndvi_change_20_24,
                'ndvi_change_2020_2024', 'NDVI_Change_2020_2024')
print("   ✓ Task submitted: ndvi_change_2020_2024")

export_to_asset(ndvi_change_total,
                'ndvi_change_total', 'NDVI_Change_Total_1990_2024')
print("   ✓ Task submitted: ndvi_change_total")

# --- Control Variables ---
export_to_asset(rainfall_change,
                'rainfall_change', 'Rainfall_Change_1990_2024')
print("   ✓ Task submitted: rainfall_change")

export_to_asset(elevation,
                'elevation', 'Elevation_SRTM')
print("   ✓ Task submitted: elevation")

export_to_asset(lci_distance,
                'lci_distance', 'LCI_Distance_Component')
print("   ✓ Task submitted: lci_distance")

# ─────────────────────────────────────────────────────────────────────
# Summary
# ─────────────────────────────────────────────────────────────────────
print("\n✅ All 13 export tasks submitted to GEE Assets!")
print("─────────────────────────────────────────────────")
print("📌 Next steps:")
print("   1. Go to code.earthengine.google.com")
print("   2. Click the 'Tasks' tab on the right panel")
print("   3. Click RUN on any UNSUBMITTED tasks")
print("   4. Wait for all tasks → COMPLETED (5–30 mins)")
print("   5. Check assets at:")
print(f"      {ASSET_PATH}/")
print("─────────────────────────────────────────────────")
print("⚠️  Keep this Colab tab open until tasks are RUNNING")

ℹ️  Folder may already exist: Cannot overwrite asset 'projects/mironga-project-marsabit/assets/marsabit'.
⏳ Submitting export tasks to GEE Assets...
   Path: projects/mironga-project-marsabit/assets/marsabit/

   ✓ Task submitted: ndvi_1990
   ✓ Task submitted: ndvi_2000
   ✓ Task submitted: ndvi_2010
   ✓ Task submitted: ndvi_2020
   ✓ Task submitted: ndvi_2024
   ✓ Task submitted: ndvi_change_1990_2000
   ✓ Task submitted: ndvi_change_2000_2010
   ✓ Task submitted: ndvi_change_2010_2020
   ✓ Task submitted: ndvi_change_2020_2024
   ✓ Task submitted: ndvi_change_total
   ✓ Task submitted: rainfall_change
   ✓ Task submitted: elevation
   ✓ Task submitted: lci_distance

✅ All 13 export tasks submitted to GEE Assets!
─────────────────────────────────────────────────
📌 Next steps:
   1. Go to code.earthengine.google.com
   2. Click the 'Tasks' tab on the right panel
   3. Click RUN on any UNSUBMITTED tasks
   4. Wait for all tasks → COMPLETED (5–30 mins)
   5. Check assets at:
      proj

In [10]:
# Check your personal GEE username
username = ee.data.getAssetRoots()
print("Your GEE asset roots:")
for root in username:
    print(f"   → {root['id']}")

Your GEE asset roots:
   → projects/mironga-project-marsabit/assets/marsabit


In [11]:
# Verify all assets were exported successfully
ASSET_PATH = 'projects/mironga-project-marsabit/assets/marsabit'

assets = ee.data.listAssets({'parent': ASSET_PATH})
asset_list = assets.get('assets', [])

expected = [
    'ndvi_1990', 'ndvi_2000', 'ndvi_2010', 'ndvi_2020', 'ndvi_2024',
    'ndvi_change_1990_2000', 'ndvi_change_2000_2010',
    'ndvi_change_2010_2020', 'ndvi_change_2020_2024',
    'ndvi_change_total', 'rainfall_change', 'elevation', 'lci_distance'
]

print(f"📦 Assets found in GEE: {len(asset_list)}/13")
print("─────────────────────────────────────────")

found_names = [a['name'].split('/')[-1] for a in asset_list]

for name in expected:
    if name in found_names:
        print(f"   ✅ {name}")
    else:
        print(f"   ❌ {name} — NOT YET EXPORTED")

print("─────────────────────────────────────────")
if len(asset_list) == 13:
    print("🎉 All 13 assets exported successfully!")
    print("   Notebook 1 is COMPLETE!")
else:
    print(f"⏳ Still waiting — {13 - len(asset_list)} assets remaining")

📦 Assets found in GEE: 13/13
─────────────────────────────────────────
   ✅ ndvi_1990
   ✅ ndvi_2000
   ✅ ndvi_2010
   ✅ ndvi_2020
   ✅ ndvi_2024
   ✅ ndvi_change_1990_2000
   ✅ ndvi_change_2000_2010
   ✅ ndvi_change_2010_2020
   ✅ ndvi_change_2020_2024
   ✅ ndvi_change_total
   ✅ rainfall_change
   ✅ elevation
   ✅ lci_distance
─────────────────────────────────────────
🎉 All 13 assets exported successfully!
   Notebook 1 is COMPLETE!


## ✅ NOTEBOOK 1 COMPLETE — Summary

All preprocessing steps have been completed successfully.

**What was accomplished:**
| Step | Task | Status |
|------|------|--------|
| Step 1 | GEE Authentication | ✅ Done |
| Step 2 | Marsabit Boundary | ✅ Done |
| Step 3 | Satellite Functions | ✅ Done |
| Step 4 | NDVI 5 Time Periods | ✅ Done |
| Step 5 | NDVI Change Maps | ✅ Done |
| Step 6 | CHIRPS Rainfall | ✅ Done |
| Step 7 | SRTM Elevation | ✅ Done |
| Step 8 | LCI Distance Component | ✅ Done |
| Step 9 | Export to GEE Assets | ✅ Done |

**All 13 assets stored at:**
projects/mironga-project-marsabit/assets/marsabit/

**Next → Notebook 2: LULC Classification**
Random Forest land cover classification for all 5 time periods.